### Import Dependencies

In [ ]:
import openai
from qdrant_client import QdrantClient

### Embedding function

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval function

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    return results

In [ ]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)

In [ ]:
retrieved_context

In [ ]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [ ]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)

In [ ]:
retrieved_context

### Format retrieved context function

In [ ]:
for id, chunk, rating in zip(retrieved_context["retrieved_context_ids"], retrieved_context["retrieved_context"], retrieved_context["retrieved_context_ratings"]):
    print(f"Context ID: {id}")
    print(f"Context: {chunk}")
    print(f"Rating: {rating}")
    print("\n")

In [ ]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [ ]:
preprocessed_context = process_context(retrieved_context)

In [ ]:
print(preprocessed_context)

### Create prompt template function

In [ ]:
def build_prompt(preprocessed_context, query):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}    
"""

    return prompt

In [ ]:
prompt = build_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summers?")

In [ ]:
print(prompt)

### Generate answer function

In [ ]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

### Combined RAG pipeline

In [ ]:
def rag_pipeline(query, top_k=5):

    retrieved_context = retrieve_data(query, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    return answer

In [ ]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

In [ ]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

In [ ]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))